# Model 2 - Skill to ESCO matcher

**Task:** unsupervised semantic retrieval / ranking. Input = one free-text course skill
string, output = ranked ESCO concepts + similarity. There are **no gold course-ESCO
labels**, so we (a) build the retriever and (b) evaluate it with the held-out **altLabel**
trick (MODEL_PLAN.md section 5).

Tiers: exact match (baseline) -> char+word TF-IDF cosine -> sentence-transformer embeddings.

> **ESCO source is a group decision (blocker in docs).** Toggle `ESCO_SOURCE` below:
> - `"full_filtered"` = skills_en.csv restricted to transversal + cross-sector (the plan's 4,241).
> - `"clean"` = skills_clean.csv as committed (2,154, no transversal level).
>
> Requires: pandas, numpy, scikit-learn (+ sentence-transformers for the embedding tier).

In [1]:
import sys, json, re
from pathlib import Path
import numpy as np
import pandas as pd

here = Path.cwd()
REPO_ROOT = here if (here / "data" / "processed").exists() else here.parent
sys.path.insert(0, str(REPO_ROOT))

from src.data_contract import (DATA_PROCESSED, COURSES_CSV, SKILLS_COL,
                               ESCO_SKILLS_FULL_CSV, ESCO_SKILLS_CLEAN_CSV, SEED)
np.random.seed(SEED)

# ---- CONFIG: which ESCO file backs the index (see blocker) ----
ESCO_SOURCE = "full_filtered"   # or "clean"
KEEP_REUSE = {"transversal", "cross-sector"}   # used only for full_filtered
print("repo root:", REPO_ROOT, "| ESCO_SOURCE =", ESCO_SOURCE)

repo root: C:\Users\jimal\02_University\Year 4\Pre-master\Software Engineering for CSAI\Software-Engineering---Group-7 | ESCO_SOURCE = full_filtered


## 1. Build the ESCO concept index

In [2]:
def norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())

if ESCO_SOURCE == "full_filtered":
    esco = pd.read_csv(ESCO_SKILLS_FULL_CSV, low_memory=False)
    esco = esco[esco["reuseLevel"].isin(KEEP_REUSE)].copy()
elif ESCO_SOURCE == "clean":
    esco = pd.read_csv(ESCO_SKILLS_CLEAN_CSV).copy()
else:
    raise ValueError(ESCO_SOURCE)

esco = esco.dropna(subset=["preferredLabel"]).reset_index(drop=True)
esco["pref_norm"] = esco["preferredLabel"].map(norm)

def alt_list(cell):
    if pd.isna(cell):
        return []
    return [norm(a) for a in str(cell).split("\n") if a.strip()]

esco["alts"] = esco["altLabels"].map(alt_list)
print(f"ESCO concepts in index: {len(esco):,}")
esco[["preferredLabel", "reuseLevel"]].head() if "reuseLevel" in esco else esco[["preferredLabel"]].head()

ESCO concepts in index: 4,241


,preferredLabel,reuseLevel
0,identify available services,cross-sector
1,perform toxicological studies,cross-sector
2,show initiative,transversal
3,apply diplomatic principles,cross-sector
4,develop energy saving concepts,cross-sector


## 2. Course skill vocabulary

The real product input: the distinct skill strings courses actually use.

In [3]:
# Source = COURSES_CSV (the cleaned, deduplicated ML-features file), not the raw
# export: it is the canonical modelling input, has higher skills fill (~40%), and
# keeps Model 2 consistent with Model 1.
courses = pd.read_csv(COURSES_CSV, low_memory=False)
vocab = set()
for cell in courses[SKILLS_COL].dropna():
    for part in str(cell).replace("\n", ",").split(","):
        p = norm(part)
        if p:
            vocab.add(p)
course_vocab = sorted(vocab)
print(f"distinct course skill strings: {len(course_vocab):,}")

distinct course skill strings: 4,229


## 3. Baseline - exact string match

The floor. Reproduces the 'why not a lookup table' finding: exact match links only a
few percent of course strings to ESCO. It also scores ~0 on the paraphrase test below,
by construction (it cannot retrieve a phrasing it has never seen).

In [4]:
esco_label_set = set(esco["pref_norm"]) | {a for alts in esco["alts"] for a in alts}
matched = sum(1 for v in course_vocab if v in esco_label_set)
exact_rate = matched / len(course_vocab)
print(f"exact-matched course strings: {matched:,} / {len(course_vocab):,} = {exact_rate:.1%}")
print(f"unmatched: {1-exact_rate:.1%}")

exact-matched course strings: 180 / 4,229 = 4.3%
unmatched: 95.7%


## 4. Held-out altLabel evaluation set

For every concept with an alt label, hold ONE out as a query whose correct answer we
know, and index the concept by its preferred label (+ remaining alts). This yields
thousands of labelled paraphrase queries at zero annotation cost (section 5).

In [5]:
rng = np.random.default_rng(SEED)
queries, gold = [], []          # query text, correct concept row-index
doc_texts = []                  # one document string per concept (index side)

for i, row in esco.iterrows():
    alts = row["alts"]
    held = None
    if alts:
        held = alts[rng.integers(len(alts))]
    remaining = [a for a in alts if a != held]
    doc_texts.append(" . ".join([row["pref_norm"], *remaining]))
    if held:
        queries.append(held)
        gold.append(i)

print(f"index documents: {len(doc_texts):,} | held-out queries: {len(queries):,}")

index documents: 4,241 | held-out queries: 4,236


In [6]:
def rank_metrics(sim_matrix, gold_idx, k=5):
    # sim_matrix: (n_queries, n_docs). gold_idx: list of correct doc indices.
    top1 = top5 = mrr = 0.0
    n = len(gold_idx)
    order = np.argsort(-sim_matrix, axis=1)[:, :k]
    for q, g in enumerate(gold_idx):
        ranked = order[q]
        if ranked[0] == g:
            top1 += 1
        if g in ranked:
            top5 += 1
            mrr += 1.0 / (list(ranked).index(g) + 1)
    return {"top1": round(top1/n, 4), "top5": round(top5/n, 4), "mrr5": round(mrr/n, 4)}

eval_results = {}

### 4a. Exact-match on the paraphrase queries (expected ~0)

In [7]:
pref_lookup = {}
for i, p in enumerate(esco["pref_norm"]):
    pref_lookup.setdefault(p, i)
hit = sum(1 for q, g in zip(queries, gold) if pref_lookup.get(q) == g)
eval_results["exact_match"] = {"top1": round(hit/len(queries), 4), "top5": None, "mrr5": None}
print("exact-match top-1 on paraphrases:", eval_results["exact_match"]["top1"],
      "(near zero by construction - that IS the finding)")

exact-match top-1 on paraphrases: 0.0 (near zero by construction - that IS the finding)


### 4b. Char + word n-gram TF-IDF cosine (no downloads)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

word_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)

Dw = word_vec.fit_transform(doc_texts); Qw = word_vec.transform(queries)
Dc = char_vec.fit_transform(doc_texts); Qc = char_vec.transform(queries)

# Combine word + char similarity (simple average).
sim = 0.5 * cosine_similarity(Qw, Dw) + 0.5 * cosine_similarity(Qc, Dc)
eval_results["tfidf_cosine"] = rank_metrics(sim, gold)
print("TF-IDF cosine:", eval_results["tfidf_cosine"])

TF-IDF cosine: {'top1': 0.758, 'top5': 0.9256, 'mrr5': 0.8261}


### 4c. Sentence-transformer embeddings (optional - needs a download)

Guarded so the notebook still runs without the package. Uncomment/install to enable.

In [9]:
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    D = model.encode(doc_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    Q = model.encode(queries, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    sim_emb = Q @ D.T
    eval_results["embeddings"] = rank_metrics(sim_emb, gold)
    print("embeddings:", eval_results["embeddings"])
except Exception as e:
    print("Skipped embedding tier:", type(e).__name__, e)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

embeddings: {'top1': 0.7297, 'top5': 0.9273, 'mrr5': 0.8098}


In [10]:
pd.DataFrame(eval_results).T

,top1,top5,mrr5
exact_match,0.0000,NaN,NaN
tfidf_cosine,0.7580,0.9256,0.8261
embeddings,0.7297,0.9273,0.8098


## 5. Behaviour on the real course vocabulary

The paraphrase test measures the method; this measures the product. For each course
skill string, take the best ESCO match and look at the similarity distribution and a
few concrete good/bad examples.

In [11]:
Vw = word_vec.transform(course_vocab); Vc = char_vec.transform(course_vocab)
vsim = 0.5 * cosine_similarity(Vw, Dw) + 0.5 * cosine_similarity(Vc, Dc)
best = vsim.argmax(axis=1); best_score = vsim.max(axis=1)

THRESHOLD = 0.35
cleared = float((best_score >= THRESHOLD).mean())
print(f"share of course strings clearing similarity >= {THRESHOLD}: {cleared:.1%}")

examples = pd.DataFrame({
    "course_skill": course_vocab,
    "best_esco": esco["preferredLabel"].values[best],
    "score": np.round(best_score, 3),
}).sort_values("score", ascending=False)
print("\nTop matches:"); display(examples.head(8))
print("\nWeak matches:"); display(examples.tail(8))

share of course strings clearing similarity >= 0.35: 61.5%

Top matches:


,course_skill,best_esco,score
1729,geometry,geometry,1.0
2394,marketing mix,marketing mix,1.0
3560,social network analysis,social network analysis,1.0
3937,typography,typography,1.0
38,accounting,accounting,1.0
515,business communication,business communication,1.0
3143,quantitative analysis,quantitative analysis,1.0
1057,data ethics,data ethics,1.0



Weak matches:


,course_skill,best_esco,score
4203,تجميع البيانات,identify available services,0.0
4202,بيانات التعريف,identify available services,0.0
1922,http,identify available services,0.0
4200,العمليات الحسابية للبيانات,identify available services,0.0
4199,العرض التقديمي,identify available services,0.0
4198,أخلاقيات البيانات,identify available services,0.0
2165,jpa,identify available services,0.0
4228,ビジュアライゼーション,identify available services,0.0


## 6. Write results to files

In [12]:
out = {
    "esco_source": ESCO_SOURCE,
    "esco_index_size": int(len(esco)),
    "distinct_course_skill_strings": len(course_vocab),
    "exact_match_rate_on_vocab": round(exact_rate, 4),
    "paraphrase_eval": eval_results,
    "course_vocab_threshold": THRESHOLD,
    "course_vocab_cleared_share": round(cleared, 4),
}
(DATA_PROCESSED / f"model2_results_{ESCO_SOURCE}.json").write_text(
    json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
examples.to_csv(DATA_PROCESSED / f"model2_match_examples_{ESCO_SOURCE}.csv", index=False, encoding="utf-8")
print(f"Wrote model2_results_{ESCO_SOURCE}.json and model2_match_examples_{ESCO_SOURCE}.csv to", DATA_PROCESSED)

Wrote model2_results_full_filtered.json and model2_match_examples_full_filtered.csv to C:\Users\jimal\02_University\Year 4\Pre-master\Software Engineering for CSAI\Software-Engineering---Group-7\data\processed
